# House Price Prediction (California Housing)
## Baseline ANN vs Neighbourhood-Feature Enhanced ANN (Optimized, Leakage-safe)

This notebook is aligned with the coursework brief:
- Regression task (predict **MedHouseVal**)
- Feedforward neural network (sklearn **MLPRegressor**) with **ReLU**
- Baseline model vs Enhanced model with neighbourhood (graph-derived) features
- Graph is used only to define neighbourhoods (kNN), **NOT** a GNN
- **No data leakage**: neighbourhood label stats use **training labels only**


## 0. Imports & Settings

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt

EARTH_RADIUS_KM = 6371.0088
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load Dataset

In [ ]:
ds = fetch_california_housing(as_frame=True)
df = ds.frame.copy()

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"].to_numpy(dtype=np.float64)

X.head(), df["MedHouseVal"].describe()


## 2. Train/Test Split (Split first to avoid leakage)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

X_train_raw.shape, X_test_raw.shape


## 3. Baseline ANN (StandardScaler + MLPRegressor)

In [ ]:
def build_ann(max_iter=2000, hidden=(128, 64), alpha=1e-4, lr=1e-3, seed=RANDOM_STATE):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=hidden,
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=lr,
            max_iter=max_iter,
            early_stopping=True,
            n_iter_no_change=20,
            random_state=seed,
            verbose=False,
        ))
    ])

def eval_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    return {"mse": float(mse), "rmse": rmse, "mae": float(mae)}

baseline_model = build_ann(max_iter=2000)
baseline_model.fit(X_train_raw, y_train)
y_pred_base = baseline_model.predict(X_test_raw)

baseline_metrics = eval_metrics(y_test, y_pred_base)
baseline_metrics


### Baseline prediction scatter

In [ ]:
plt.figure()
plt.scatter(y_test, y_pred_base, s=6)
plt.xlabel("True MedHouseVal")
plt.ylabel("Predicted MedHouseVal (Baseline)")
plt.title("Baseline ANN: True vs Predicted")
plt.show()


## 4. Neighbourhood (Graph) Features

We define neighbourhoods using **k-nearest neighbours** on (Latitude, Longitude).

### Distance
We use **Haversine distance** (great-circle distance) for more realistic geographic proximity.

### Leakage control
- Neighbours are always selected among **training samples only**.
- Any label-based statistic uses **y_train only** (never y_test).

In [ ]:
def coords_radians(df_):
    coords = df_[["Latitude", "Longitude"]].to_numpy(dtype=np.float64)
    return np.radians(coords)

def add_neighbour_features(
    df_,                 # target (train or test)
    ref_df,              # reference pool (must be X_train_raw)
    ref_y,               # reference labels (must be y_train)
    k=25,
    exclude_self=False,  # True for training set (when df_ is the same pool as ref_df)
    add_ref_feature_means=("MedInc",),  # optional extra: neighbour mean of non-label cols
    eps_km=1e-3,
):
    if k <= 0:
        raise ValueError("k must be positive")

    ref_coords = coords_radians(ref_df)
    tgt_coords = coords_radians(df_)

    n_neighbors = k + 1 if exclude_self else k
    nn = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="haversine",
        algorithm="ball_tree",
    )
    nn.fit(ref_coords)

    dist_rad, idx = nn.kneighbors(tgt_coords, return_distance=True)

    if exclude_self:
        idx = idx[:, 1:k+1]
        dist_rad = dist_rad[:, 1:k+1]
    else:
        idx = idx[:, :k]
        dist_rad = dist_rad[:, :k]

    dist_km = dist_rad * EARTH_RADIUS_KM
    nb_y = ref_y[idx]  # (n, k) from y_train only

    nb_mean_price = nb_y.mean(axis=1)
    nb_std_price = nb_y.std(axis=1)
    nb_mean_dist_km = dist_km.mean(axis=1)

    w = 1.0 / (dist_km + eps_km)
    nb_wmean_price = (w * nb_y).sum(axis=1) / w.sum(axis=1)

    out = df_.copy()
    out["nb_mean_price"] = nb_mean_price
    out["nb_std_price"] = nb_std_price
    out["nb_mean_dist_km"] = nb_mean_dist_km
    out["nb_wmean_price"] = nb_wmean_price
    out["nb_count"] = float(k)

    # Optional: mean of other (non-label) neighbour features
    for col in add_ref_feature_means:
        if col in ref_df.columns:
            ref_col = ref_df[col].to_numpy(dtype=np.float64)
            out[f"nb_mean_{col}"] = ref_col[idx].mean(axis=1)

    return out

k = 25
X_train_enh = add_neighbour_features(
    X_train_raw, X_train_raw, y_train, k=k, exclude_self=True,
    add_ref_feature_means=("MedInc", "AveRooms")
)
X_test_enh = add_neighbour_features(
    X_test_raw, X_train_raw, y_train, k=k, exclude_self=False,
    add_ref_feature_means=("MedInc", "AveRooms")
)

X_train_enh.shape, X_test_enh.shape, X_train_enh.filter(like="nb_").head()


## 5. Enhanced ANN (same model class, more input features)

In [ ]:
enhanced_model = build_ann(max_iter=2000)
enhanced_model.fit(X_train_enh, y_train)
y_pred_enh = enhanced_model.predict(X_test_enh)

enhanced_metrics = eval_metrics(y_test, y_pred_enh)
enhanced_metrics


### Enhanced prediction scatter

In [ ]:
plt.figure()
plt.scatter(y_test, y_pred_enh, s=6)
plt.xlabel("True MedHouseVal")
plt.ylabel("Predicted MedHouseVal (Enhanced)")
plt.title("Enhanced ANN: True vs Predicted")
plt.show()


## 6. Baseline vs Enhanced Comparison

In [ ]:
def print_compare(base, enh):
    improvement = (base["mse"] - enh["mse"]) / max(base["mse"], 1e-12) * 100.0
    out = pd.DataFrame([base, enh], index=["Baseline", "Enhanced"])
    out["mse_improvement_%"] = [np.nan, improvement]
    return out

compare_df = print_compare(baseline_metrics, enhanced_metrics)
compare_df


## 7. Optional: Sweep k

This can help you find a reasonable neighbourhood size (bias–variance trade-off).

In [ ]:
def run_k_sweep(k_list=(5,10,25,50,100), epochs=2000):
    rows = []
    base_model = build_ann(max_iter=epochs)
    base_model.fit(X_train_raw, y_train)
    base_pred = base_model.predict(X_test_raw)
    base_m = eval_metrics(y_test, base_pred)

    for k in tqdm(k_list):
        Xtr = add_neighbour_features(X_train_raw, X_train_raw, y_train, k=k, exclude_self=True,
                                     add_ref_feature_means=("MedInc",))
        Xte = add_neighbour_features(X_test_raw, X_train_raw, y_train, k=k, exclude_self=False,
                                     add_ref_feature_means=("MedInc",))
        m = build_ann(max_iter=epochs)
        m.fit(Xtr, y_train)
        pred = m.predict(Xte)
        met = eval_metrics(y_test, pred)
        rows.append({"k": k, **met})
    return base_m, pd.DataFrame(rows).sort_values("mse")

# Uncomment to run (can take a few minutes depending on k_list / epochs)
# base_m, sweep_df = run_k_sweep(k_list=(5,10,25,50,100), epochs=2000)
# print("Baseline:", base_m)
# sweep_df


## 8. Discussion prompts (for your report)

Even if you don't write the report, these are the key points your team can discuss:

- **Why neighbourhood features help**: spatial autocorrelation; neighbour price acts like label smoothing.
- **Potential over-optimism** with random split: train/test may be geographically close (spatial leakage in evaluation, not label leakage).
- **Choice of k**: bias–variance trade-off; too small = noisy, too large = oversmoothing.
- **Target cap at 5.0** in this dataset: right-censoring; affects error distribution and interpretation.
- **Distance metric**: Euclidean degrees vs Haversine km; why Haversine is more meaningful.
- **Limitations**: dependence on training coverage; performance may drop for sparse regions or distribution shift.
